In [40]:
from pyspark.sql import SparkSession

# Создание Spark сессии
spark = SparkSession.builder \
    .appName("MinimalSparkSession") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "1") \
    .getOrCreate()

from pyspark.sql.functions import col
from pyspark.sql.functions import *
from pyspark.sql.window import Window

### Создаем датафреймы

In [41]:
users = spark.createDataFrame(
[
("u1", "Berlin"),
("u2", "Berlin"),
("u3", "Munich"),
("u4", "Hamburg"), ],
["user_id", "city"] )

orders = spark.createDataFrame(
[
("o1", "u1", "p1", 2, 10.0),
("o2", "u1", "p2", 1, 30.0),
("o3", "u2", "p1", 1, 10.0),
("o4", "u2", "p3", 5, 7.0),
("o5", "u3", "p2", 3, 30.0),
("o6", "u3", "p3", 1, 7.0),
("o7", "u4", "p1", 10, 10.0), ],
["order_id", "user_id", "product_id", "qty", "price"] )

products = spark.createDataFrame(
[
("p1", "Ring VOLA"),
("p2", "Ring POROG"),
("p3", "Ring TISHINA"), ],
["product_id", "product_name"])

### Шаг 1. Расчета столбца revenue

In [42]:
orders = orders.withColumn('revenue', col('qty') * col('price'))
orders.show()

+--------+-------+----------+---+-----+-------+
|order_id|user_id|product_id|qty|price|revenue|
+--------+-------+----------+---+-----+-------+
|      o1|     u1|        p1|  2| 10.0|   20.0|
|      o2|     u1|        p2|  1| 30.0|   30.0|
|      o3|     u2|        p1|  1| 10.0|   10.0|
|      o4|     u2|        p3|  5|  7.0|   35.0|
|      o5|     u3|        p2|  3| 30.0|   90.0|
|      o6|     u3|        p3|  1|  7.0|    7.0|
|      o7|     u4|        p1| 10| 10.0|  100.0|
+--------+-------+----------+---+-----+-------+



### Шаг 2. Джойны

In [43]:
sales = users.join(orders, 'user_id', 'inner')\
            .join(products, 'product_id', 'inner')
sales.show()

+----------+-------+-------+--------+---+-----+-------+------------+
|product_id|user_id|   city|order_id|qty|price|revenue|product_name|
+----------+-------+-------+--------+---+-----+-------+------------+
|        p1|     u4|Hamburg|      o7| 10| 10.0|  100.0|   Ring VOLA|
|        p1|     u2| Berlin|      o3|  1| 10.0|   10.0|   Ring VOLA|
|        p1|     u1| Berlin|      o1|  2| 10.0|   20.0|   Ring VOLA|
|        p2|     u3| Munich|      o5|  3| 30.0|   90.0|  Ring POROG|
|        p2|     u1| Berlin|      o2|  1| 30.0|   30.0|  Ring POROG|
|        p3|     u3| Munich|      o6|  1|  7.0|    7.0|Ring TISHINA|
|        p3|     u2| Berlin|      o4|  5|  7.0|   35.0|Ring TISHINA|
+----------+-------+-------+--------+---+-----+-------+------------+



### Шаг 3. Расчет метрик

In [44]:
metrics = sales.groupby('city', 'product_id', 'product_name')\
                .agg(
                    count('order_id').alias('orders_cnt'),\
                     sum('qty').alias('qty_sum'),\
                     sum('revenue').alias('revenue_sum')
                    )
metrics.show()

+-------+----------+------------+----------+-------+-----------+
|   city|product_id|product_name|orders_cnt|qty_sum|revenue_sum|
+-------+----------+------------+----------+-------+-----------+
|Hamburg|        p1|   Ring VOLA|         1|     10|      100.0|
| Berlin|        p1|   Ring VOLA|         2|      3|       30.0|
| Munich|        p2|  Ring POROG|         1|      3|       90.0|
| Berlin|        p2|  Ring POROG|         1|      1|       30.0|
| Munich|        p3|Ring TISHINA|         1|      1|        7.0|
| Berlin|        p3|Ring TISHINA|         1|      5|       35.0|
+-------+----------+------------+----------+-------+-----------+



### Шаг 4. Расчет top-2

In [45]:
window_city  = Window.partitionBy('city').orderBy(col('revenue_sum').desc())
top2_city_products = metrics.withColumn('rn', row_number().over(window_city)) \
                            .filter(col('rn')<=2) \
                            .drop('rn')
top2_city_products.show()

+-------+----------+------------+----------+-------+-----------+
|   city|product_id|product_name|orders_cnt|qty_sum|revenue_sum|
+-------+----------+------------+----------+-------+-----------+
| Berlin|        p3|Ring TISHINA|         1|      5|       35.0|
| Berlin|        p1|   Ring VOLA|         2|      3|       30.0|
|Hamburg|        p1|   Ring VOLA|         1|     10|      100.0|
| Munich|        p2|  Ring POROG|         1|      3|       90.0|
| Munich|        p3|Ring TISHINA|         1|      1|        7.0|
+-------+----------+------------+----------+-------+-----------+



### Шаг 5. Записываем в HDFS

In [46]:
top2_city_products.write \
                    .mode('overwrite') \
                    .parquet('hdfs://namenode:9000/user/top2_city_products.parquet')
print("Файл успешно записан в HDFS как Parquet")

Файл успешно записан в HDFS как Parquet


### Шаг 6. Читаем файл из HDFS

In [47]:
df_read = spark.read.parquet('hdfs://namenode:9000/user/top2_city_products.parquet')
df_read.show()

+-------+----------+------------+----------+-------+-----------+
|   city|product_id|product_name|orders_cnt|qty_sum|revenue_sum|
+-------+----------+------------+----------+-------+-----------+
| Berlin|        p3|Ring TISHINA|         1|      5|       35.0|
| Berlin|        p1|   Ring VOLA|         2|      3|       30.0|
|Hamburg|        p1|   Ring VOLA|         1|     10|      100.0|
| Munich|        p2|  Ring POROG|         1|      3|       90.0|
| Munich|        p3|Ring TISHINA|         1|      1|        7.0|
+-------+----------+------------+----------+-------+-----------+



In [48]:
spark.stop()